In [ ]:
from google.colab import files
uploaded = files.upload()


Saving olist_customers_dataset.csv to olist_customers_dataset.csv
Saving olist_products_dataset.csv to olist_products_dataset.csv
Saving olist_sellers_dataset.csv to olist_sellers_dataset.csv
Saving product_category_name_translation.csv to product_category_name_translation.csv
Saving olist_geolocation_dataset.csv to olist_geolocation_dataset.csv
Saving olist_order_items_dataset.csv to olist_order_items_dataset.csv
Saving olist_order_payments_dataset.csv to olist_order_payments_dataset.csv
Saving olist_order_reviews_dataset.csv to olist_order_reviews_dataset.csv
Saving olist_orders_dataset.csv to olist_orders_dataset.csv


In [ ]:
import pandas as pd

orders = pd.read_csv('olist_orders_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
category_translation = pd.read_csv('product_category_name_translation.csv')

In [ ]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


## 1.1 dataset analysis

In [ ]:
# ORDERS DATASET ANALYSIS
# This is the central table — every other table links back to order_id here

print("=== ORDERS ===")

# .shape returns (number_of_rows, number_of_columns) as a tuple
print("Shape (rows, cols):", orders.shape)

# .isnull() marks every cell True/False depending on whether it's missing
# .sum() then adds up the True values per column (True=1, False=0)
# The [...> 0] filter only shows columns that actually HAVE missing values,
# so clean columns don't clutter the output
print("Missing values per column:\n", orders.isnull().sum()[orders.isnull().sum() > 0])

# .duplicated() flags rows that are exact copies of an earlier row
# .sum() counts how many such duplicate rows exist
print("Duplicate rows:", orders.duplicated().sum())

# Bonus check specific to this file: order_id should be 100% unique
# since it's the primary key other tables join on
print("Is order_id fully unique?", orders['order_id'].is_unique)

# See what values exist in order_status — helps us understand
# WHY some delivery date columns have missing values (e.g. canceled orders
# never get a delivered_customer_date)
print("Order status breakdown:\n", orders['order_status'].value_counts())

=== ORDERS ===
Shape (rows, cols): (99441, 8)
Missing values per column:
 order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
Duplicate rows: 0
Is order_id fully unique? True
Order status breakdown:
 order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


## 1.2 file



In [ ]:
# CUSTOMERS DATASET ANALYSIS
# Links to orders via customer_id — tells us WHERE (city/state) each order came from

print("=== CUSTOMERS ===")

print("Shape (rows, cols):", customers.shape)

print("Missing values per column:\n", customers.isnull().sum()[customers.isnull().sum() > 0])

print("Duplicate rows:", customers.duplicated().sum())

# customer_id should be unique — it's the join key to orders
print("Is customer_id fully unique?", customers['customer_id'].is_unique)

# customer_unique_id is DIFFERENT from customer_id — one real person can have
# multiple customer_id's (one per order) but the same customer_unique_id.
# Comparing the two counts tells us how many REPEAT customers exist.
print("Unique customer_id count:", customers['customer_id'].nunique())
print("Unique customer_unique_id count:", customers['customer_unique_id'].nunique())

# Top 10 states by customer count — quick regional signal
print("Top 10 states:\n", customers['customer_state'].value_counts().head(10))

=== CUSTOMERS ===
Shape (rows, cols): (99441, 5)
Missing values per column:
 Series([], dtype: int64)
Duplicate rows: 0
Is customer_id fully unique? True
Unique customer_id count: 99441
Unique customer_unique_id count: 96096
Top 10 states:
 customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64


## 1.3 file ORDER_ITEMS DATASET ANALYSIS

In [ ]:
# ORDER_ITEMS DATASET ANALYSIS
# One row per item in an order (an order with 3 products = 3 rows here)
# This is the key REVENUE table — has price and freight_value

print("=== ORDER_ITEMS ===")

print("Shape (rows, cols):", order_items.shape)

print("Missing values per column:\n", order_items.isnull().sum()[order_items.isnull().sum() > 0])

print("Duplicate rows:", order_items.duplicated().sum())

# order_id is NOT expected to be unique here (one order can have multiple items)
# so instead check: how many unique orders are represented?
print("Unique order_id count:", order_items['order_id'].nunique())
print("Total rows (order-item lines):", len(order_items))

# Quick revenue signal
print("Total revenue (price sum):", order_items['price'].sum())
print("Total freight cost:", order_items['freight_value'].sum())
print("Average items per order:", len(order_items) / order_items['order_id'].nunique())

=== ORDER_ITEMS ===
Shape (rows, cols): (112650, 7)
Missing values per column:
 Series([], dtype: int64)
Duplicate rows: 0
Unique order_id count: 98666
Total rows (order-item lines): 112650
Total revenue (price sum): 13591643.700000003
Total freight cost: 2251909.54
Average items per order: 1.1417306873695092


## 1.4 file _ ORDER_PAYMENTS DATASET ANALYSIS

In [ ]:
# ORDER_PAYMENTS DATASET ANALYSIS
# One row per payment method used on an order
# (some orders split payment across multiple methods, e.g. part credit card + part voucher)

print("=== PAYMENTS ===")

print("Shape (rows, cols):", payments.shape)

print("Missing values per column:\n", payments.isnull().sum()[payments.isnull().sum() > 0])

print("Duplicate rows:", payments.duplicated().sum())

# Like order_items, order_id won't be unique here if payments were split
print("Unique order_id count:", payments['order_id'].nunique())
print("Total rows (payment lines):", len(payments))

# Payment method breakdown — useful KPI for the dashboard
print("Payment type breakdown:\n", payments['payment_type'].value_counts())

# Installments — common in Brazil, worth understanding distribution
print("Installments summary:\n", payments['payment_installments'].describe())

# Total payment value — should roughly match order_items total (price+freight)
print("Total payment value:", payments['payment_value'].sum())

=== PAYMENTS ===
Shape (rows, cols): (103886, 5)
Missing values per column:
 Series([], dtype: int64)
Duplicate rows: 0
Unique order_id count: 99440
Total rows (payment lines): 103886
Payment type breakdown:
 payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64
Installments summary:
 count    103886.000000
mean          2.853349
std           2.687051
min           0.000000
25%           1.000000
50%           1.000000
75%           4.000000
max          24.000000
Name: payment_installments, dtype: float64
Total payment value: 16008872.12


## 1.5 ORDER_REVIEWS DATASET ANALYSIS

In [ ]:
# ORDER_REVIEWS DATASET ANALYSIS
# One row per review — but we already know some orders have >1 review

print("=== REVIEWS ===")

print("Shape (rows, cols):", reviews.shape)

print("Missing values per column:\n", reviews.isnull().sum()[reviews.isnull().sum() > 0])

print("Duplicate rows:", reviews.duplicated().sum())

print("Unique order_id count:", reviews['order_id'].nunique())
print("Total rows (review lines):", len(reviews))

# Orders with more than 1 review — flagged earlier, confirm the count
multi_review_orders = reviews['order_id'].value_counts()
print("Orders with more than 1 review:", (multi_review_orders > 1).sum())

# Review score distribution — key satisfaction KPI
print("Review score breakdown:\n", reviews['review_score'].value_counts().sort_index())
print("Average review score:", reviews['review_score'].mean())

=== REVIEWS ===
Shape (rows, cols): (99224, 7)
Missing values per column:
 review_comment_title      87656
review_comment_message    58247
dtype: int64
Duplicate rows: 0
Unique order_id count: 98673
Total rows (review lines): 99224
Orders with more than 1 review: 547
Review score breakdown:
 review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64
Average review score: 4.08642062404257


## 1.6. PRODUCTS DATASET ANALYSIS

In [ ]:
# PRODUCTS DATASET ANALYSIS
# Links to order_items via product_id — gives category, weight, dimensions

print("=== PRODUCTS ===")

print("Shape (rows, cols):", products.shape)

print("Missing values per column:\n", products.isnull().sum()[products.isnull().sum() > 0])

print("Duplicate rows:", products.duplicated().sum())

# product_id should be unique — it's the join key
print("Is product_id fully unique?", products['product_id'].is_unique)

# Top 10 categories by product count
print("Top 10 categories:\n", products['product_category_name'].value_counts().head(10))

# How many distinct categories exist overall?
print("Total unique categories:", products['product_category_name'].nunique())

=== PRODUCTS ===
Shape (rows, cols): (32951, 9)
Missing values per column:
 product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
Duplicate rows: 0
Is product_id fully unique? True
Top 10 categories:
 product_category_name
cama_mesa_banho           3029
esporte_lazer             2867
moveis_decoracao          2657
beleza_saude              2444
utilidades_domesticas     2335
automotivo                1900
informatica_acessorios    1639
brinquedos                1411
relogios_presentes        1329
telefonia                 1134
Name: count, dtype: int64
Total unique categories: 73


## 1.7 SELLERS DATASET ANALYSIS

In [ ]:
# SELLERS DATASET ANALYSIS
# Links to order_items via seller_id — tells us WHO fulfilled each order and FROM WHERE

print("=== SELLERS ===")

print("Shape (rows, cols):", sellers.shape)

print("Missing values per column:\n", sellers.isnull().sum()[sellers.isnull().sum() > 0])

print("Duplicate rows:", sellers.duplicated().sum())

# seller_id should be unique — it's the join key
print("Is seller_id fully unique?", sellers['seller_id'].is_unique)

# Top 10 states by seller count — compare this to customer states later
# (is supply concentrated in the same regions as demand?)
print("Top 10 seller states:\n", sellers['seller_state'].value_counts().head(10))

=== SELLERS ===
Shape (rows, cols): (3095, 4)
Missing values per column:
 Series([], dtype: int64)
Duplicate rows: 0
Is seller_id fully unique? True
Top 10 seller states:
 seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
Name: count, dtype: int64


## 1.8 CATEGORY_TRANSLATION DATASET ANALYSIS

In [ ]:
# CATEGORY_TRANSLATION DATASET ANALYSIS
# Simple lookup table: Portuguese category name -> English category name
# Used only to make product_category_name readable in the dashboard

print("=== CATEGORY_TRANSLATION ===")

print("Shape (rows, cols):", category_translation.shape)

print("Missing values per column:\n", category_translation.isnull().sum()[category_translation.isnull().sum() > 0])

print("Duplicate rows:", category_translation.duplicated().sum())

# Sanity check: does every category in products actually have a translation?
categories_in_products = products['product_category_name'].dropna().unique()
categories_in_translation = category_translation['product_category_name'].unique()

missing_translations = set(categories_in_products) - set(categories_in_translation)
print("Product categories missing a translation:", missing_translations)

=== CATEGORY_TRANSLATION ===
Shape (rows, cols): (71, 2)
Missing values per column:
 Series([], dtype: int64)
Duplicate rows: 0
Product categories missing a translation: {'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}


1. Delivery-time KPIs must filter to order_status == 'delivered'

2. ~3.5% repeat customer rate; SP dominates both demand (42%) and supply (60%) — worth testing if that supply concentration affects delivery time in other states
3. 775 orders have no line items — decide join type carefully
~R$165K mismatch between payments total and order_items total — needs investigation
4. Review sentiment is bimodal, not gradual (mostly 5-star or 1-star)
5. 547 orders have multiple reviews — need a dedup rule
6. 2 product categories have no English translation — need manual mapping

# REFERENTIAL INTEGRITY CHECK

In [ ]:
# REFERENTIAL INTEGRITY CHECK
# Confirms that "key" columns linking tables actually match up correctly.
# A result of 1.0 (100%) means every row has a valid match — safe to join.
# Anything less means some rows would be silently dropped or mismatched in a join.

print("=== REFERENTIAL INTEGRITY CHECKS ===")

# order_items -> orders (every item should belong to a real order)
print("order_items -> orders match rate:", order_items['order_id'].isin(orders['order_id']).mean())

# payments -> orders
print("payments -> orders match rate:", payments['order_id'].isin(orders['order_id']).mean())

# reviews -> orders
print("reviews -> orders match rate:", reviews['order_id'].isin(orders['order_id']).mean())

# order_items -> products (every item should reference a real product)
print("order_items -> products match rate:", order_items['product_id'].isin(products['product_id']).mean())

# order_items -> sellers (every item should reference a real seller)
print("order_items -> sellers match rate:", order_items['seller_id'].isin(sellers['seller_id']).mean())

# orders -> customers
print("orders -> customers match rate:", orders['customer_id'].isin(customers['customer_id']).mean())

print()
print("=== REVERSE CHECK: does every order have items/payments? ===")
# We already suspected 775 orders have no order_items — confirm it here
print("orders -> order_items match rate:", orders['order_id'].isin(order_items['order_id']).mean())
print("orders -> payments match rate:", orders['order_id'].isin(payments['order_id']).mean())
print("orders -> reviews match rate:", orders['order_id'].isin(reviews['order_id']).mean())

=== REFERENTIAL INTEGRITY CHECKS ===
order_items -> orders match rate: 1.0
payments -> orders match rate: 1.0
reviews -> orders match rate: 1.0
order_items -> products match rate: 1.0
order_items -> sellers match rate: 1.0
orders -> customers match rate: 1.0

=== REVERSE CHECK: does every order have items/payments? ===
orders -> order_items match rate: 0.9922064339658692
orders -> payments match rate: 0.9999899437857624
orders -> reviews match rate: 0.9922768274655324


"why," not just the "what."

## Why we're running this at all

In a multi-table dataset like Olist, each table only makes sense connected to the others — `order_items` alone doesn't tell you the customer, `reviews` alone doesn't tell you the delivery time. The whole value of your project depends on **joining these tables together correctly** in SQL and Power BI later.

But joins can silently go wrong. If `order_items` has a `product_id` that doesn't actually exist in `products`, a join won't throw an error — it'll just quietly produce a blank or drop that row, and you'd never know unless you specifically checked. That's dangerous because a wrong number in your final dashboard (say, understated revenue) with no warning is worse than a program crashing loudly.

**Referential integrity checking = proactively testing every relationship before you build on top of it**, so you're not debugging phantom errors three steps later in Power BI.

## What the code actually does — line by line logic

Take this line:
```python
order_items['order_id'].isin(orders['order_id']).mean()
```

- `order_items['order_id']` → grab the column of order IDs from the items table
- `.isin(orders['order_id'])` → for **each** order_id in that column, check: "does this value exist anywhere in the orders table's order_id column?" → returns `True`/`False` for every single row
- `.mean()` → since `True`=1 and `False`=0, averaging them gives you **the percentage of rows that found a match**

So `1.0` = 100% of rows matched = every single order_item correctly points to a real order. `0.95` would mean 5% of order_items reference an order_id that doesn't exist anywhere — a real problem.

## Why we did it in *both* directions

This is the part that's easy to miss. There are two separate questions:

1. **"Does every child record have a valid parent?"** → `order_items → orders` (the first 6 checks) — this checks *outward*, from the smaller/detail table back to its parent
2. **"Does every parent record have a child?"** → `orders → order_items` (the reverse checks) — this checks the opposite direction: does every order actually have at least one item/payment/review?

These aren't redundant — they answer different questions. An order_id in `order_items` might always point to something valid (#1 = 1.0), while at the same time, some orders in the `orders` table might have **zero** matching items (#2 < 1.0). Both were true in your results.

## What your specific output meant, plainly

- **First 6 results = 1.0** → every child row (item, payment, review) points to a real parent. No broken/orphan references anywhere. This means: **safe to join in either direction with confidence** — nothing will silently vanish due to a bad key.
- **Reverse checks < 1.0** → not every order has a matching item/payment/review. Specifically ~0.78% of orders have no items, ~0.001% (1 order) has no payment, ~0.77% have no review. This isn't a "bug" — it reflects real business reality (some orders got canceled before an item was even shipped, not everyone leaves a review).

**Why this matters practically:** it tells you which type of SQL join to use later. If you use an `INNER JOIN` from `orders` to `order_items`, those ~775 incomplete orders will just disappear from your results without warning — fine if that's what you want, bad if you actually needed to report on all orders including canceled ones. Knowing this *now* means you make that decision deliberately instead of accidentally.

# Step_2 :- Clean & Prepare Data in Excel

# 2.1 Filter orders to delivered status
Create a filtered version of orders where order_status == 'delivered' only. This becomes your base table for any delivery-time KPI, since canceled/processing orders don't have real delivery dates.

In [ ]:
# CLEANING STEP 1: FILTER TO DELIVERED ORDERS
# Why: Only 'delivered' orders have real delivery dates.
# Canceled/processing/shipped orders have blank delivery timestamps
# because the delivery never happened (or hasn't happened yet) —
# including them in delivery-time math would either error out or skew averages.

# Create a NEW dataframe, don't overwrite the original 'orders'
# (we may still need the full orders table for order-count/status analysis later)
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

# .copy() ensures this is a fully independent dataframe, not just a "view"
# of the original — avoids a common pandas warning/bug later when we add columns to it

print("Original orders count:", len(orders))
print("Delivered orders count:", len(orders_delivered))
print("Filtered out:", len(orders) - len(orders_delivered), "non-delivered orders")

# Sanity check: delivered orders should now have almost NO missing delivery dates
print("\nMissing values in delivered-only orders:")
print(orders_delivered.isnull().sum()[orders_delivered.isnull().sum() > 0])

Original orders count: 99441
Delivered orders count: 96478
Filtered out: 2963 non-delivered orders

Missing values in delivered-only orders:
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
dtype: int64


In [ ]:
# Drop the 24 'delivered' orders that still have missing timestamp data
# These are inconsistent records — status says delivered, but the dates don't back it up
# At only 24 rows (0.02% of delivered orders), dropping is safe and won't bias results

orders_delivered_clean = orders_delivered.dropna(
    subset=['order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date']
).copy()

print("Before dropping inconsistent rows:", len(orders_delivered))
print("After dropping inconsistent rows:", len(orders_delivered_clean))

# Confirm zero missing values now
print("\nRemaining missing values:")
print(orders_delivered_clean.isnull().sum()[orders_delivered_clean.isnull().sum() > 0])

Before dropping inconsistent rows: 96478
After dropping inconsistent rows: 96455

Remaining missing values:
Series([], dtype: int64)


# 2.2 :- Handle missing product categories
For the 610 products missing category_name (and related fields), decide and apply a rule — either drop these rows or label them 'unknown'. We'll label rather than drop, since dropping loses valid revenue data tied to those products.


In [ ]:
# Convert date columns from text to actual datetime objects so we can do date math
orders_delivered_clean['order_purchase_timestamp'] = pd.to_datetime(orders_delivered_clean['order_purchase_timestamp'])
orders_delivered_clean['order_delivered_customer_date'] = pd.to_datetime(orders_delivered_clean['order_delivered_customer_date'])
orders_delivered_clean['order_estimated_delivery_date'] = pd.to_datetime(orders_delivered_clean['order_estimated_delivery_date'])

# Actual delivery time: how many days from purchase to actual delivery
orders_delivered_clean['delivery_days_actual'] = (
    orders_delivered_clean['order_delivered_customer_date'] - orders_delivered_clean['order_purchase_timestamp']
).dt.days

# Delivery delay: positive = late, negative = early/on-time
# (actual delivered date MINUS estimated date)
orders_delivered_clean['delivery_delay_days'] = (
    orders_delivered_clean['order_delivered_customer_date'] - orders_delivered_clean['order_estimated_delivery_date']
).dt.days

print("Average actual delivery time (days):", orders_delivered_clean['delivery_days_actual'].mean())
print("Average delivery delay (days):", orders_delivered_clean['delivery_delay_days'].mean())
print("% of orders delivered LATE:", (orders_delivered_clean['delivery_delay_days'] > 0).mean() * 100)
print("% of orders delivered EARLY or ON TIME:", (orders_delivered_clean['delivery_delay_days'] <= 0).mean() * 100)

Average actual delivery time (days): 12.093100409517392
Average delivery delay (days): -11.874584002902909
% of orders delivered LATE: 6.7731066300347305
% of orders delivered EARLY or ON TIME: 93.22689336996527


Orders take ~12 days on average from purchase to delivery
On average, deliveries arrive ~12 days before the estimated delivery date (delay = -11.87) — meaning Olist sets fairly conservative/padded delivery estimates
93.2% of orders arrive early or on time, and only 6.8% arrive late

In [ ]:
# Isolate only the late-delivered orders (delay > 0)
late_orders = orders_delivered_clean[orders_delivered_clean['delivery_delay_days'] > 0]

print("Number of late orders:", len(late_orders))
print("Average delay among LATE orders only (days):", late_orders['delivery_delay_days'].mean())

# Merge with customers to see which states have the most late deliveries
late_with_state = late_orders.merge(customers, on='customer_id', how='left')
print("\nTop 10 states by late delivery count:")
print(late_with_state['customer_state'].value_counts().head(10))

Number of late orders: 6533
Average delay among LATE orders only (days): 10.620848002449105

Top 10 states by late delivery count:
customer_state
SP    1819
RJ    1495
MG     519
BA     396
RS     325
SC     291
ES     214
PR     199
CE     176
PE     153
Name: count, dtype: int64


In [ ]:
# Calculate late delivery RATE per state, not just raw count
# This normalizes for the fact that SP simply has more orders overall

# Total delivered orders per state
total_by_state = orders_delivered_clean.merge(customers, on='customer_id', how='left')['customer_state'].value_counts()

# Late orders per state (from previous step)
late_by_state = late_with_state['customer_state'].value_counts()

# Combine into a rate: late_count / total_count * 100
delay_rate = (late_by_state / total_by_state * 100).sort_values(ascending=False)

print("Late delivery RATE by state (%), top 15:")
print(delay_rate.head(15))

print("\nFor comparison, SP's rate specifically:")
print("SP late rate:", delay_rate['SP'], "%")

Late delivery RATE by state (%), top 15:
customer_state
AL    21.410579
MA    17.458101
SE    15.223881
PI    13.865546
CE    13.771518
RR    12.195122
BA    12.162162
RJ    12.107224
PA    11.205074
ES    10.726817
PB    10.444874
TO     9.854015
MS     9.700428
PE     9.604520
RN     9.282700
Name: count, dtype: float64

For comparison, SP's rate specifically:
SP late rate: 4.492689191859316 %


# This is a strong, real insight for your case study:

"States with low seller density face dramatically higher delivery delay rates — AL customers experience late deliveries over 1 in 5 times, roughly 3x the rate of seller-dense states. This suggests fulfillment/logistics infrastructure, not raw distance alone, is a key delay driver — a finding directly actionable for logistics planning."

# Confirmed — this is a genuinely powerful finding.

SP's late rate: 4.49% vs. AL's late rate: 21.41% — that's nearly 5x higher in Alagoas compared to São Paulo. The hypothesis holds up completely: states where sellers are concentrated (SP, and by extension nearby RJ/MG/PR) get dramatically more reliable delivery, while remote/low-seller-density states get hit hard.

This is now a clean, three-part story for your case study:

60% of sellers are based in SP, but only 42% of customers are
SP customers get a 4.5% late rate — among the best in the dataset
Remote states without local sellers (AL, MA, SE, PI, CE) see late rates of 12-21% — 3-5x worse

This directly answers one of your four core business questions ("where are delivery delays concentrated") with a specific, actionable root cause (seller geographic concentration) rather than a vague "delivery is sometimes late." This is dashboard-map material — a state-by-state late-rate map would make this insight immediately visible to leadership.

In [ ]:
# CLEANING STEP 2: HANDLE MISSING PRODUCT CATEGORIES
# 610 products have no category_name (and related fields are blank too)
# Decision: label as 'unknown' rather than drop — these products still have
# valid orders/revenue attached to them in order_items, so dropping would
# silently lose real transaction data from our revenue analysis

products_clean = products.copy()

# Fill missing category name with a clear placeholder label
products_clean['product_category_name'] = products_clean['product_category_name'].fillna('unknown')

# Confirm the fix
print("Missing category_name after fix:", products_clean['product_category_name'].isnull().sum())
print("\nHow many products are now labeled 'unknown':", (products_clean['product_category_name'] == 'unknown').sum())

# Note: name_length, description_length, photos_qty are still blank for these rows
# That's fine — those aren't needed for category-based revenue analysis
print("\nOther still-missing columns (expected, not category-critical):")
print(products_clean.isnull().sum()[products_clean.isnull().sum() > 0])

Missing category_name after fix: 0

How many products are now labeled 'unknown': 610

Other still-missing columns (expected, not category-critical):
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


In [ ]:
# Check the 2 products missing weight/dimensions - are they real products
# with orders attached, or edge cases we can safely ignore?

missing_dims = products_clean[products_clean['product_weight_g'].isnull()]
print(missing_dims[['product_id', 'product_category_name']])

# Do these products actually appear in order_items? If not, doesn't matter at all
print("\nDo these product_ids appear in order_items?")
print(order_items['product_id'].isin(missing_dims['product_id']).sum(), "matching rows")

                             product_id product_category_name
8578   09ff539a621711667c43eba6a3bd8466                 bebes
18851  5eb564652db742ff8f28759cd8d2652a               unknown

Do these product_ids appear in order_items?
18 matching rows


In [ ]:
# Fill missing weight/dimensions using the AVERAGE for that product's category
# This is more accurate than a flat 0 or a dataset-wide average,
# since a "bebes" (baby products) item and a "moveis" (furniture) item
# have very different typical weights

for col in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
    # Calculate average value per category (excluding nulls)
    category_avg = products_clean.groupby('product_category_name')[col].transform('mean')
    # Fill only the missing values with that category's average
    products_clean[col] = products_clean[col].fillna(category_avg)

# Confirm the fix
print("Remaining missing values in products_clean:")
print(products_clean.isnull().sum()[products_clean.isnull().sum() > 0])


Remaining missing values in products_clean:
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
dtype: int64


# 2.1 :- step _3 Task 3: Translate categories to English,

In [ ]:
# CLEANING STEP 3: TRANSLATE CATEGORY NAMES TO ENGLISH
# We already found 2 categories with no match in category_translation:
# 'pc_gamer' and 'portateis_cozinha_e_preparadores_de_alimentos'
# We'll manually add those 2 mappings before merging, so nothing ends up blank

category_translation_clean = category_translation.copy()

# Manually add the 2 missing mappings as new rows
missing_map = pd.DataFrame({
    'product_category_name': ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'],
    'product_category_name_english': ['pc_gaming', 'kitchen_portables_food_prep']
})
category_translation_clean = pd.concat([category_translation_clean, missing_map], ignore_index=True)

# Also add a mapping for our 'unknown' label so it translates cleanly too
unknown_map = pd.DataFrame({
    'product_category_name': ['unknown'],
    'product_category_name_english': ['unknown']
})
category_translation_clean = pd.concat([category_translation_clean, unknown_map], ignore_index=True)

# Now merge English names into products_clean
products_clean = products_clean.merge(category_translation_clean, on='product_category_name', how='left')

# Confirm: are there ANY blanks left in the English category column?
print("Missing English category names:", products_clean['product_category_name_english'].isnull().sum())

# Preview
print(products_clean[['product_category_name', 'product_category_name_english']].drop_duplicates().head(10))

Missing English category names: 0
   product_category_name product_category_name_english
0             perfumaria                     perfumery
1                  artes                           art
2          esporte_lazer                sports_leisure
3                  bebes                          baby
4  utilidades_domesticas                    housewares
5  instrumentos_musicais           musical_instruments
6             cool_stuff                    cool_stuff
7       moveis_decoracao               furniture_decor
8       eletrodomesticos               home_appliances
9             brinquedos                          toys


##2.4_step_4 :- Task 4 :- Deduplicate the 547 multi-review orders
Deduplicate multi-review ordersFor the 547 orders with more than one review, apply a rule — keep the most recent review per order (by review_creation_date) — so joining reviews to orders doesn't duplicate rows.

In [ ]:
# CLEANING STEP 4: DEDUPLICATE MULTI-REVIEW ORDERS
# 547 orders have more than 1 review. If we join reviews to orders as-is,
# those orders would appear twice (or more) in our merged data, inflating counts.
# Rule: keep only the MOST RECENT review per order (by review_creation_date)

reviews_clean = reviews.copy()

# Convert review_creation_date to actual datetime so we can sort by it properly
reviews_clean['review_creation_date'] = pd.to_datetime(reviews_clean['review_creation_date'])

# Sort so the most recent review comes first within each order_id group
reviews_clean = reviews_clean.sort_values('review_creation_date', ascending=False)

# Keep only the FIRST row per order_id (which is now the most recent, thanks to the sort)
reviews_clean = reviews_clean.drop_duplicates(subset='order_id', keep='first')

# Confirm: order_id should now be fully unique
print("Original reviews count:", len(reviews))
print("Deduplicated reviews count:", len(reviews_clean))
print("Is order_id now fully unique?", reviews_clean['order_id'].is_unique)

Original reviews count: 99224
Deduplicated reviews count: 98673
Is order_id now fully unique? True


# step_6 extra distribution

In [ ]:
# TASK 6a: BUILD SALES MASTER TABLE
# Base = orders (LEFT JOIN outward) so we keep ALL orders, even edge cases
# This table answers revenue/sales trend questions by category, region, time

# Start from full orders table (not just delivered — we want all orders for sales context)
sales_master = orders.copy()

# LEFT JOIN order_items (adds price, freight, product_id, seller_id)
sales_master = sales_master.merge(order_items, on='order_id', how='left')

# LEFT JOIN products_clean (adds English category name)
sales_master = sales_master.merge(
    products_clean[['product_id', 'product_category_name_english']],
    on='product_id', how='left'
)

# LEFT JOIN customers (adds customer state/city)
sales_master = sales_master.merge(
    customers[['customer_id', 'customer_city', 'customer_state']],
    on='customer_id', how='left'
)

# LEFT JOIN payments (adds payment_type, payment_value)
# Note: payments can have multiple rows per order (split payments) —
# for now we'll just merge, and handle aggregation later if needed
sales_master = sales_master.merge(
    payments[['order_id', 'payment_type', 'payment_value']],
    on='order_id', how='left'
)

print("Sales master shape:", sales_master.shape)
print("\nColumns:", list(sales_master.columns))
print("\nSample rows:")
print(sales_master.head(3))

# Sanity check: total row count should be close to order_items count (~112,650)
# since one order can have multiple items, and now potentially multiple payments too

Sales master shape: (118434, 19)

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name_english', 'customer_city', 'customer_state', 'payment_type', 'payment_value']

Sample rows:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
2  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
2    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   

  order_

In [ ]:
# FIX: Aggregate payments to ONE row per order BEFORE merging
# This collapses split payments (e.g. credit_card + voucher) into a single summary row per order,
# so we don't fan-out and duplicate item/price rows

payments_agg = payments.groupby('order_id').agg(
    total_payment_value=('payment_value', 'sum'),
    payment_type_primary=('payment_type', 'first'),   # just take the first payment method used
    num_payment_methods=('payment_type', 'nunique')
).reset_index()

# Rebuild sales_master WITHOUT the row-level payments merge
sales_master = orders.copy()
sales_master = sales_master.merge(order_items, on='order_id', how='left')
sales_master = sales_master.merge(
    products_clean[['product_id', 'product_category_name_english']], on='product_id', how='left'
)
sales_master = sales_master.merge(
    customers[['customer_id', 'customer_city', 'customer_state']], on='customer_id', how='left'
)
# Now merge the AGGREGATED payments (1 row per order) instead of raw payments
sales_master = sales_master.merge(payments_agg, on='order_id', how='left')

print("Corrected sales master shape:", sales_master.shape)
print("Should now be close to order_items count (112,650):", len(order_items))

# Verify: does this one order now appear only once per item (not fanned out)?
print("\nSame order check:")
print(sales_master[sales_master['order_id'] == 'e481f51cbdc54678b7cc49136f2d6af7'])

Corrected sales master shape: (113425, 20)
Should now be close to order_items count (112,650): 112650

Same order check:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   

  order_estimated_delivery_date  order_item_id  \
0           2017-10-18 00:00:00            1.0   

                         product_id                         seller_id  \
0  87285b34884572647811a353c7ac498a  3504c0cb71d7fa48d967e0e4c94d59d9   

   shipping_limit_date  price  freight_value product_category_name_english  \
0  2017-10-06 11:07:15  29.99           8.72                    housewares   

  customer_city customer_state  total_payment_value payment_type_prim

# full Step_6_ task 2.6 :-creating master table


In [ ]:
import pandas as pd

# ============================================================
# TASK 6: BUILD FINAL CLEAN MASTER TABLES
# Assumes these already exist from earlier steps:
# orders, order_items, payments, reviews, products, customers,
# sellers, category_translation, orders_delivered_clean,
# products_clean, reviews_clean
# ============================================================

# ------------------------------------------------------------
# 6a. SALES MASTER TABLE
# Base = orders (ALL orders, LEFT JOIN outward)
# Used for: revenue trends by category/region/time
# ------------------------------------------------------------

# Aggregate payments to 1 row per order FIRST (avoids fan-out/duplication)
payments_agg = payments.groupby('order_id').agg(
    total_payment_value=('payment_value', 'sum'),
    payment_type_primary=('payment_type', 'first'),
    num_payment_methods=('payment_type', 'nunique')
).reset_index()

sales_master = orders.copy()
sales_master = sales_master.merge(order_items, on='order_id', how='left')
sales_master = sales_master.merge(
    products_clean[['product_id', 'product_category_name_english']],
    on='product_id', how='left'
)
sales_master = sales_master.merge(
    customers[['customer_id', 'customer_city', 'customer_state']],
    on='customer_id', how='left'
)
sales_master = sales_master.merge(payments_agg, on='order_id', how='left')

print("=== SALES MASTER ===")
print("Shape:", sales_master.shape)
print("Columns:", list(sales_master.columns))
print()

# ------------------------------------------------------------
# 6b. DELIVERY + SATISFACTION TABLE
# Base = orders_delivered_clean, INNER JOIN reviews_clean
# Used for: delivery delay vs review score analysis
# ------------------------------------------------------------

delivery_satisfaction = orders_delivered_clean.merge(
    reviews_clean[['order_id', 'review_score', 'review_creation_date']],
    on='order_id', how='inner'
)

print("=== DELIVERY + SATISFACTION ===")
print("Shape:", delivery_satisfaction.shape)
correlation = delivery_satisfaction['delivery_delay_days'].corr(delivery_satisfaction['review_score'])
print("Correlation (delay vs review score):", correlation)
print()

# ------------------------------------------------------------
# 6c. SELLER PERFORMANCE SCORECARD
# Base = order_items, LEFT JOIN delivery info + reviews
# Used for: seller-level KPIs (volume, delay, rating, revenue)
# ------------------------------------------------------------

seller_scorecard = order_items.merge(
    orders_delivered_clean[['order_id', 'delivery_delay_days']],
    on='order_id', how='left'
)
seller_scorecard = seller_scorecard.merge(
    reviews_clean[['order_id', 'review_score']],
    on='order_id', how='left'
)
seller_scorecard = seller_scorecard.merge(
    sellers[['seller_id', 'seller_city', 'seller_state']],
    on='seller_id', how='left'
)

seller_summary = seller_scorecard.groupby('seller_id').agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('price', 'sum'),
    avg_delivery_delay=('delivery_delay_days', 'mean'),
    avg_review_score=('review_score', 'mean'),
    seller_state=('seller_state', 'first')
).reset_index()

print("=== SELLER SCORECARD ===")
print("Shape:", seller_summary.shape)
print(seller_summary.head())
print()

# ------------------------------------------------------------
# EXPORT all three clean tables as CSV — ready for SQL import
# ------------------------------------------------------------

sales_master.to_csv('sales_master.csv', index=False)
delivery_satisfaction.to_csv('delivery_satisfaction.csv', index=False)
seller_summary.to_csv('seller_scorecard.csv', index=False)

print("All 3 master tables exported as CSV.")

=== SALES MASTER ===
Shape: (113425, 20)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name_english', 'customer_city', 'customer_state', 'total_payment_value', 'payment_type_primary', 'num_payment_methods']

=== DELIVERY + SATISFACTION ===
Shape: (95809, 12)
Correlation (delay vs review score): -0.26736419030292513

=== SELLER SCORECARD ===
Shape: (3095, 6)
                          seller_id  total_orders  total_revenue  \
0  0015a82c2db000af6aaaf3ae2ecb0532             3        2685.00   
1  001cca7ae9ae17fb1caed9dfb1094831           200       25080.03   
2  001e6ad469a905060d959994f1b41e4f             1         250.00   
3  002100f778ceb8431b7a1020ff7ab48f            51        1234.50   
4  003554e2dce176b5555353e4f3555ac8

# step 6 explanation
Sales master: 113,425 rows, 20 columns — clean, one row per order-item, no fan-out. Ready for revenue/category/region analysis.

Delivery + satisfaction: 95,809 rows (delivered orders with a matching review), and the correlation is -0.267. This directly answers business question #3: there IS a measurable negative relationship between delivery delay and review score — the more delayed an order, the lower the rating tends to be. -0.267 is a moderate (not huge) correlation, which is an honest, defensible finding — not every unhappy review is about delivery, but delay clearly moves the needle. Good line for your case study: "Delivery delay shows a moderate negative correlation (-0.27) with review score, confirming delivery performance measurably impacts customer satisfaction — though it isn't the only driver."

Seller scorecard: 3,095 sellers, all with total orders, revenue, avg delay, avg review score, and state. Notice seller 001e6ad4... has avg_delivery_delay = NaN despite having 1 order and a review score of 1.0 — that's a seller whose single order likely wasn't in orders_delivered_clean (maybe canceled, or one of the 23 inconsistent rows we dropped earlier), but still got reviewed. Worth a note in your log — a handful of sellers may show NaN delay for this reason; not an error, just a side effect of the join logic, and fine to leave as-is since averaging automatically ignores NaNs.

In [ ]:
from google.colab import files
files.download('sales_master.csv')
files.download('delivery_satisfaction.csv')
files.download('seller_scorecard.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>